# LLM4ChipDesign: ATTRITION Hardware Trojan Analysis

Welcome to the LLM4ChipDesign undergraduate course notebook! This comprehensive guide will introduce you to the ATTRITION project, which focuses on hardware Trojan detection and analysis in chip designs. 

## Course Objectives
1. Understand hardware security threats and hardware Trojans
2. Learn about the ATTRITION research framework
3. Analyze original vs Trojan-infested chip designs
4. Apply LLM techniques for chip design analysis
5. Implement basic hardware design validation tools

## Dataset Overview
The ATTRITION project contains:
- **Original Designs**: Clean hardware designs including AES, GPS, MIPS processors
- **HT-Infested Designs**: Hardware Trojan infected versions of the original designs
- **Analysis Tools**: Scripts for comparing and analyzing design differences

Let's begin our journey into hardware security and LLM applications in chip design!

## Section 1: Environment Setup and Dependencies

First, let's install and import all the required libraries for our LLM4ChipDesign workflow.

In [ ]:
# Install required dependencies for Google Colab
!pip install transformers torch matplotlib networkx plotly pandas seaborn
!pip install gym==0.21.0  # Required by ATTRITION project
!pip install pyverilog    # For Verilog parsing

# Download ATTRITION project (if running on Colab)
import os
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('ATTRITION-main'):
        !wget -q https://github.com/DfX-NYUAD/ATTRITION/archive/refs/heads/main.zip
        !unzip -q main.zip
        !rm main.zip
    os.chdir('ATTRITION-main')
    
print("Environment setup complete!")

In [ ]:
# Import required libraries
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import networkx as nx
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

## Section 2: LLM Model Loading and Configuration

In this section, we'll set up language models that can be used for chip design analysis and code generation tasks.

In [ ]:
# Load pre-trained language models for code analysis
from transformers import GPT2LMHeadModel, GPT2Tokenizer, pipeline

# Initialize a lightweight model for demonstration
try:
    # Use a smaller model for Colab compatibility
    model_name = "distilgpt2"
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name)
    
    # Set up text generation pipeline
    text_generator = pipeline("text-generation", 
                             model=model, 
                             tokenizer=tokenizer, 
                             max_length=200,
                             temperature=0.7)
    
    print(f"Successfully loaded {model_name}")
    print(f"Model parameters: {model.num_parameters():,}")
    
except Exception as e:
    print(f"Error loading model: {e}")
    print("Continuing with mock functions for demonstration...")

## Section 3: Chip Design Data Preparation

Now let's explore the ATTRITION dataset and prepare the hardware design data for analysis.

In [ ]:
# Explore the ATTRITION project structure
def explore_attrition_structure():
    """Explore and display the ATTRITION project structure"""
    
    # Define paths
    original_path = "original_files"
    ht_infested_path = "HT_infested_designs"
    
    print("ATTRITION Project Structure Analysis")
    print("="*50)
    
    # Check original files
    if os.path.exists(original_path):
        original_files = [f for f in os.listdir(original_path) if f.endswith('.v')]
        print(f"\nOriginal Design Files ({len(original_files)}):")
        for i, file in enumerate(original_files[:10], 1):  # Show first 10
            print(f"  {i}. {file}")
        if len(original_files) > 10:
            print(f"  ... and {len(original_files) - 10} more files")
    
    # Check HT-infested designs
    if os.path.exists(ht_infested_path):
        ht_dirs = [d for d in os.listdir(ht_infested_path) if os.path.isdir(os.path.join(ht_infested_path, d))]
        print(f"\nHardware Trojan Infested Design Sets ({len(ht_dirs)}):")
        for i, dir_name in enumerate(ht_dirs[:5], 1):  # Show first 5
            dir_path = os.path.join(ht_infested_path, dir_name)
            file_count = len([f for f in os.listdir(dir_path) if f.endswith('.v')])
            print(f"  {i}. {dir_name} ({file_count} files)")
    
    return original_files if os.path.exists(original_path) else []

# Run the exploration
design_files = explore_attrition_structure()

In [ ]:
# Verilog design analysis functions
class VerilogAnalyzer:
    """A class to analyze Verilog design files"""
    
    def __init__(self):
        self.stats = defaultdict(int)
    
    def parse_verilog_file(self, filepath, max_lines=1000):
        """Parse a Verilog file and extract basic statistics"""
        if not os.path.exists(filepath):
            return None
            
        stats = {
            'filename': os.path.basename(filepath),
            'total_lines': 0,
            'wire_declarations': 0,
            'gate_instances': 0,
            'modules': 0,
            'unique_gates': set(),
            'file_size_kb': os.path.getsize(filepath) / 1024
        }
        
        try:
            with open(filepath, 'r') as f:
                for line_num, line in enumerate(f):
                    if line_num >= max_lines:  # Limit for large files
                        break
                    stats['total_lines'] += 1
                    line = line.strip()
                    
                    # Count wire declarations
                    if line.startswith('wire '):
                        stats['wire_declarations'] += 1
                    
                    # Count module declarations
                    if line.startswith('module '):
                        stats['modules'] += 1
                    
                    # Count gate instances (simplified)
                    if any(gate in line for gate in ['_X1 ', '_X2 ', '_X4 ', 'BUF_', 'INV_', 'AND_', 'OR_', 'NAND_', 'NOR_']):
                        stats['gate_instances'] += 1
                        # Extract gate type
                        for gate_type in ['BUF_X1', 'INV_X1', 'AND2_X1', 'OR2_X1', 'NAND2_X1', 'NOR2_X1']:
                            if gate_type in line:
                                stats['unique_gates'].add(gate_type)
                                break
        
        except Exception as e:
            print(f"Error parsing {filepath}: {e}")
            return None
            
        stats['unique_gates'] = list(stats['unique_gates'])
        return stats
    
    def analyze_design_set(self, file_list, max_files=5):
        """Analyze a set of design files"""
        results = []
        for i, filename in enumerate(file_list[:max_files]):
            filepath = os.path.join("original_files", filename)
            stats = self.parse_verilog_file(filepath)
            if stats:
                results.append(stats)
                print(f"Analyzed: {filename} ({stats['file_size_kb']:.1f} KB)")
        return results

# Initialize analyzer and test with original designs
analyzer = VerilogAnalyzer()
print("Verilog Analyzer initialized!")

# Test with first few original designs
if design_files:
    analysis_results = analyzer.analyze_design_set(design_files, max_files=3)
    print(f"\nAnalyzed {len(analysis_results)} design files")

## Section 4: Verilog Code Generation with LLM

Let's use our LLM to generate and analyze Verilog code patterns commonly found in chip designs.

In [ ]:
# LLM-based Verilog code generation
class VerilogGenerator:
    """Generate and analyze Verilog code using LLM"""
    
    def __init__(self, text_generator=None):
        self.text_generator = text_generator
        self.verilog_templates = {
            'adder': """// 4-bit Adder Module
module adder_4bit(
    input [3:0] a,
    input [3:0] b,
    input cin,
    output [3:0] sum,
    output cout
);
    assign {cout, sum} = a + b + cin;
endmodule""",
            
            'mux': """// 4-to-1 Multiplexer
module mux_4to1(
    input [1:0] sel,
    input [3:0] data_in,
    output reg data_out
);
    always @(*) begin
        case(sel)
            2'b00: data_out = data_in[0];
            2'b01: data_out = data_in[1];
            2'b10: data_out = data_in[2];
            2'b11: data_out = data_in[3];
            default: data_out = 1'b0;
        endcase
    end
endmodule""",
            
            'fsm': """// Simple State Machine
module simple_fsm(
    input clk,
    input reset,
    input trigger,
    output reg [1:0] state_out
);
    reg [1:0] current_state, next_state;
    
    parameter IDLE = 2'b00,
              ACTIVE = 2'b01,
              DONE = 2'b10;
    
    always @(posedge clk or posedge reset) begin
        if (reset)
            current_state <= IDLE;
        else
            current_state <= next_state;
    end
    
    always @(*) begin
        case(current_state)
            IDLE: next_state = trigger ? ACTIVE : IDLE;
            ACTIVE: next_state = DONE;
            DONE: next_state = IDLE;
            default: next_state = IDLE;
        endcase
    end
    
    assign state_out = current_state;
endmodule"""
        }
    
    def generate_verilog_prompt(self, design_type, description):
        """Generate a prompt for Verilog code generation"""
        prompt = f"""Generate a Verilog module for a {design_type}.
Description: {description}
Requirements: Include proper input/output declarations, module structure, and comments.

module"""
        return prompt
    
    def generate_code(self, prompt):
        """Generate Verilog code using LLM or templates"""
        if self.text_generator:
            try:
                result = self.text_generator(prompt, max_length=150, num_return_sequences=1)
                return result[0]['generated_text']
            except:
                # Fallback to template
                return self.get_template_code('adder')
        else:
            # Use templates for demonstration
            return self.get_template_code('adder')
    
    def get_template_code(self, template_name):
        """Get predefined template code"""
        return self.verilog_templates.get(template_name, "// Template not found")
    
    def demonstrate_generation(self):
        """Demonstrate different Verilog generation patterns"""
        print("Verilog Code Generation Examples")
        print("="*40)
        
        for template_name, code in self.verilog_templates.items():
            print(f"\n{template_name.upper()} MODULE:")
            print("-" * 20)
            print(code[:200] + "..." if len(code) > 200 else code)

# Initialize generator and demonstrate
verilog_gen = VerilogGenerator(text_generator if 'text_generator' in locals() else None)
verilog_gen.demonstrate_generation()

## Section 5: HDL Syntax Validation

Now we'll implement functions to validate the syntax and structure of generated Verilog code.

In [ ]:
# HDL Syntax Validation Tools
class VerilogValidator:
    """Validate Verilog code syntax and structure"""
    
    def __init__(self):
        self.keywords = ['module', 'endmodule', 'input', 'output', 'reg', 'wire', 
                        'always', 'assign', 'if', 'else', 'case', 'endcase', 
                        'begin', 'end', 'posedge', 'negedge']
        
        self.operators = ['&', '|', '^', '~', '&&', '||', '!', '+', '-', '*', 
                         '/', '%', '==', '!=', '<', '>', '<=', '>=', '<<', '>>', 
                         '?', ':', '=', '<=']
    
    def check_syntax(self, verilog_code):
        """Perform basic syntax checking on Verilog code"""
        errors = []
        warnings = []
        
        lines = verilog_code.split('\n')
        
        # Check for basic structure
        has_module = False
        has_endmodule = False
        paren_count = 0
        brace_count = 0
        
        for i, line in enumerate(lines, 1):
            line = line.strip()
            if not line or line.startswith('//'):
                continue
            
            # Check for module declaration
            if line.startswith('module '):
                has_module = True
            if line.strip() == 'endmodule':
                has_endmodule = True
            
            # Check parentheses balance
            paren_count += line.count('(') - line.count(')')
            brace_count += line.count('{') - line.count('}')
            
            # Check for semicolon at end of statements
            if any(keyword in line for keyword in ['assign', 'wire', 'reg']) and not line.endswith(';'):
                if not line.endswith(',') and not line.endswith('('):
                    warnings.append(f"Line {i}: Missing semicolon")
        
        # Structural checks
        if not has_module:
            errors.append("Missing module declaration")
        if not has_endmodule:
            errors.append("Missing endmodule")
        if paren_count != 0:
            errors.append(f"Unbalanced parentheses (count: {paren_count})")
        if brace_count != 0:
            errors.append(f"Unbalanced braces (count: {brace_count})")
        
        return {
            'errors': errors,
            'warnings': warnings,
            'is_valid': len(errors) == 0
        }
    
    def analyze_complexity(self, verilog_code):
        """Analyze the complexity of Verilog code"""
        lines = verilog_code.split('\n')
        metrics = {
            'total_lines': len([l for l in lines if l.strip() and not l.strip().startswith('//')]),
            'comment_lines': len([l for l in lines if l.strip().startswith('//')]),
            'always_blocks': verilog_code.count('always'),
            'assignments': verilog_code.count('assign'),
            'conditionals': verilog_code.count('if') + verilog_code.count('case'),
            'complexity_score': 0
        }
        
        # Simple complexity score calculation
        metrics['complexity_score'] = (metrics['always_blocks'] * 3 + 
                                     metrics['conditionals'] * 2 + 
                                     metrics['assignments'])
        
        return metrics

# Test the validator with generated code
validator = VerilogValidator()

# Test with adder template
adder_code = verilog_gen.get_template_code('adder')
print("ADDER MODULE VALIDATION:")
print("="*30)
validation_result = validator.check_syntax(adder_code)
complexity = validator.analyze_complexity(adder_code)

print(f"Valid: {validation_result['is_valid']}")
print(f"Errors: {len(validation_result['errors'])}")
print(f"Warnings: {len(validation_result['warnings'])}")
print(f"Complexity Score: {complexity['complexity_score']}")

if validation_result['errors']:
    print("Errors found:")
    for error in validation_result['errors']:
        print(f"  - {error}")

if validation_result['warnings']:
    print("Warnings:")
    for warning in validation_result['warnings']:
        print(f"  - {warning}")

## Section 6: Design Rule Checking Implementation

Let's implement basic design rule checking algorithms to verify hardware design constraints.

In [ ]:
# Design Rule Checking (DRC) Implementation
class DesignRuleChecker:
    """Implement basic design rule checking for hardware designs"""
    
    def __init__(self):
        self.rules = {
            'max_fanout': 10,
            'max_gate_delay': 100,  # picoseconds
            'min_wire_width': 1,    # arbitrary units
            'max_power': 1000       # mW
        }
    
    def check_fanout(self, netlist_data):
        """Check fanout violations"""
        violations = []
        # Simplified fanout check - count outputs per signal
        signal_fanouts = defaultdict(int)
        
        for line in netlist_data.split('\n'):
            if '.Z(' in line:  # Output connections
                match = re.search(r'\.Z\(([^)]+)\)', line)
                if match:
                    signal = match.group(1)
                    signal_fanouts[signal] += 1
        
        for signal, fanout in signal_fanouts.items():
            if fanout > self.rules['max_fanout']:
                violations.append({
                    'type': 'fanout',
                    'signal': signal,
                    'value': fanout,
                    'limit': self.rules['max_fanout']
                })
        
        return violations
    
    def estimate_timing(self, gate_count, logic_depth_estimate=10):
        """Estimate basic timing characteristics"""
        # Simplified timing estimation
        gate_delay = 20  # picoseconds per gate
        wire_delay = 5   # picoseconds per wire
        
        estimated_delay = (logic_depth_estimate * gate_delay) + (gate_count * 0.1 * wire_delay)
        
        return {
            'estimated_delay_ps': estimated_delay,
            'logic_depth': logic_depth_estimate,
            'timing_violation': estimated_delay > self.rules['max_gate_delay']
        }
    
    def estimate_power(self, gate_count, frequency_mhz=100):
        """Estimate power consumption"""
        # Simplified power estimation
        power_per_gate = 0.1  # mW per gate
        dynamic_power = gate_count * power_per_gate * (frequency_mhz / 100)
        static_power = gate_count * 0.01  # Static leakage
        
        total_power = dynamic_power + static_power
        
        return {
            'dynamic_power_mw': dynamic_power,
            'static_power_mw': static_power,
            'total_power_mw': total_power,
            'power_violation': total_power > self.rules['max_power']
        }
    
    def comprehensive_check(self, verilog_content, gate_count=1000):
        """Perform comprehensive design rule checking"""
        results = {
            'fanout_violations': self.check_fanout(verilog_content),
            'timing_analysis': self.estimate_timing(gate_count),
            'power_analysis': self.estimate_power(gate_count),
            'gate_count': gate_count
        }
        
        # Overall compliance
        results['overall_compliant'] = (
            len(results['fanout_violations']) == 0 and
            not results['timing_analysis']['timing_violation'] and
            not results['power_analysis']['power_violation']
        )
        
        return results

# Test DRC with sample data
drc = DesignRuleChecker()

# Create sample netlist data for testing
sample_netlist = """
BUF_X1 U1 ( .A(n1), .Z(n2) );
BUF_X1 U2 ( .A(n2), .Z(n3) );
AND2_X1 U3 ( .A1(n2), .A2(n4), .Z(n5) );
OR2_X1 U4 ( .A1(n2), .A2(n6), .Z(n7) );
"""

print("DESIGN RULE CHECKING EXAMPLE:")
print("="*35)

# Run comprehensive check
drc_results = drc.comprehensive_check(sample_netlist, gate_count=150)

print(f"Gate Count: {drc_results['gate_count']}")
print(f"Overall Compliant: {drc_results['overall_compliant']}")
print(f"Fanout Violations: {len(drc_results['fanout_violations'])}")
print(f"Estimated Delay: {drc_results['timing_analysis']['estimated_delay_ps']:.1f} ps")
print(f"Total Power: {drc_results['power_analysis']['total_power_mw']:.2f} mW")

if drc_results['fanout_violations']:
    print("\nFanout Violations:")
    for violation in drc_results['fanout_violations'][:3]:
        print(f"  Signal {violation['signal']}: {violation['value']} > {violation['limit']}")

## Section 7: Performance Metrics Evaluation

Let's develop metrics to evaluate the quality of generated designs including area, timing, and power analysis.

In [ ]:
# Performance Metrics Evaluation System
class PerformanceMetrics:
    """Evaluate design performance across multiple dimensions"""
    
    def __init__(self):
        self.gate_library = {
            'BUF_X1': {'area': 1.0, 'delay': 15, 'power': 0.05},
            'INV_X1': {'area': 0.8, 'delay': 12, 'power': 0.04},
            'AND2_X1': {'area': 1.2, 'delay': 18, 'power': 0.06},
            'OR2_X1': {'area': 1.2, 'delay': 18, 'power': 0.06},
            'NAND2_X1': {'area': 1.0, 'delay': 15, 'power': 0.05},
            'NOR2_X1': {'area': 1.0, 'delay': 15, 'power': 0.05},
        }
    
    def analyze_gate_distribution(self, verilog_content):
        """Analyze distribution of different gate types"""
        gate_counts = defaultdict(int)
        
        for line in verilog_content.split('\n'):
            for gate_type in self.gate_library.keys():
                if gate_type in line and 'U' in line:  # Gate instance
                    gate_counts[gate_type] += 1
        
        total_gates = sum(gate_counts.values())
        gate_distribution = {gate: count/total_gates*100 if total_gates > 0 else 0 
                           for gate, count in gate_counts.items()}
        
        return gate_counts, gate_distribution
    
    def estimate_area(self, gate_counts):
        """Estimate total chip area"""
        total_area = 0
        area_breakdown = {}
        
        for gate_type, count in gate_counts.items():
            if gate_type in self.gate_library:
                gate_area = self.gate_library[gate_type]['area'] * count
                area_breakdown[gate_type] = gate_area
                total_area += gate_area
        
        return total_area, area_breakdown
    
    def estimate_critical_path_delay(self, gate_counts, avg_logic_depth=8):
        """Estimate critical path delay"""
        if not gate_counts:
            return 0, {}
        
        # Weighted average delay calculation
        total_delay_weight = 0
        total_gates = 0
        
        for gate_type, count in gate_counts.items():
            if gate_type in self.gate_library:
                delay = self.gate_library[gate_type]['delay']
                total_delay_weight += delay * count
                total_gates += count
        
        avg_gate_delay = total_delay_weight / total_gates if total_gates > 0 else 20
        critical_path_delay = avg_gate_delay * avg_logic_depth
        
        return critical_path_delay, {'avg_gate_delay': avg_gate_delay, 'logic_depth': avg_logic_depth}
    
    def estimate_total_power(self, gate_counts, frequency_mhz=100, activity_factor=0.15):
        """Estimate total power consumption"""
        dynamic_power = 0
        static_power = 0
        power_breakdown = {}
        
        for gate_type, count in gate_counts.items():
            if gate_type in self.gate_library:
                base_power = self.gate_library[gate_type]['power']
                gate_dynamic_power = base_power * count * activity_factor * (frequency_mhz / 100)
                gate_static_power = base_power * count * 0.1  # 10% static
                
                power_breakdown[gate_type] = {
                    'dynamic': gate_dynamic_power,
                    'static': gate_static_power,
                    'total': gate_dynamic_power + gate_static_power
                }
                
                dynamic_power += gate_dynamic_power
                static_power += gate_static_power
        
        return {
            'dynamic_power_mw': dynamic_power,
            'static_power_mw': static_power,
            'total_power_mw': dynamic_power + static_power,
            'breakdown': power_breakdown
        }
    
    def comprehensive_analysis(self, verilog_content):
        """Perform comprehensive performance analysis"""
        print("COMPREHENSIVE PERFORMANCE ANALYSIS")
        print("="*40)
        
        # Gate analysis
        gate_counts, gate_distribution = self.analyze_gate_distribution(verilog_content)
        print(f"Total Gates: {sum(gate_counts.values())}")
        
        # Area analysis
        total_area, area_breakdown = self.estimate_area(gate_counts)
        print(f"Total Area: {total_area:.2f} units²")
        
        # Timing analysis
        critical_path_delay, timing_details = self.estimate_critical_path_delay(gate_counts)
        print(f"Critical Path Delay: {critical_path_delay:.1f} ps")
        
        # Power analysis
        power_analysis = self.estimate_total_power(gate_counts)
        print(f"Total Power: {power_analysis['total_power_mw']:.3f} mW")
        
        return {
            'gate_counts': gate_counts,
            'gate_distribution': gate_distribution,
            'area': {'total': total_area, 'breakdown': area_breakdown},
            'timing': {'critical_path_delay': critical_path_delay, 'details': timing_details},
            'power': power_analysis
        }

# Test performance metrics
perf_metrics = PerformanceMetrics()

# Test with sample design
sample_design = """
module test_design(input a, b, output y);
BUF_X1 U1 ( .A(a), .Z(n1) );
BUF_X1 U2 ( .A(b), .Z(n2) );
AND2_X1 U3 ( .A1(n1), .A2(n2), .Z(n3) );
INV_X1 U4 ( .A(n3), .Z(y) );
endmodule
"""

# Run comprehensive analysis
results = perf_metrics.comprehensive_analysis(sample_design)

# Show gate distribution
print(f"\nGate Distribution:")
for gate, percentage in results['gate_distribution'].items():
    if percentage > 0:
        print(f"  {gate}: {percentage:.1f}%")

## Section 8: Visualization of Generated Designs

Let's create visualization tools to display generated circuit structures, timing diagrams, and design hierarchy.

In [ ]:
# Visualization Tools for Hardware Designs
class DesignVisualizer:
    """Create visualizations for hardware designs"""
    
    def __init__(self):
        self.colors = plt.cm.Set3(np.linspace(0, 1, 12))
    
    def plot_gate_distribution(self, gate_counts, title="Gate Distribution"):
        """Plot gate type distribution"""
        if not gate_counts:
            print("No gate data to visualize")
            return
        
        gates = list(gate_counts.keys())
        counts = list(gate_counts.values())
        
        plt.figure(figsize=(10, 6))
        bars = plt.bar(gates, counts, color=self.colors[:len(gates)])
        plt.title(title)
        plt.xlabel('Gate Type')
        plt.ylabel('Count')
        plt.xticks(rotation=45)
        
        # Add count labels on bars
        for bar, count in zip(bars, counts):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    str(count), ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
    
    def plot_performance_metrics(self, perf_data):
        """Plot comprehensive performance metrics"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Design Performance Metrics', fontsize=16)
        
        # Gate distribution pie chart
        ax1 = axes[0, 0]
        if perf_data['gate_counts']:
            gates = list(perf_data['gate_counts'].keys())
            counts = list(perf_data['gate_counts'].values())
            ax1.pie(counts, labels=gates, autopct='%1.1f%%', startangle=90)
            ax1.set_title('Gate Type Distribution')
        
        # Area breakdown
        ax2 = axes[0, 1]
        if perf_data['area']['breakdown']:
            areas = list(perf_data['area']['breakdown'].values())
            gates = list(perf_data['area']['breakdown'].keys())
            bars = ax2.bar(gates, areas, color=self.colors[:len(gates)])
            ax2.set_title('Area Breakdown (units²)')
            ax2.set_ylabel('Area')
            plt.setp(ax2.get_xticklabels(), rotation=45)
        
        # Power breakdown
        ax3 = axes[1, 0]
        power_data = perf_data['power']
        power_types = ['Dynamic', 'Static']
        power_values = [power_data['dynamic_power_mw'], power_data['static_power_mw']]
        ax3.pie(power_values, labels=power_types, autopct='%1.2f mW', startangle=90)
        ax3.set_title('Power Distribution')
        
        # Performance summary
        ax4 = axes[1, 1]
        metrics = ['Area (units²)', 'Delay (ps)', 'Power (mW)', 'Gates']
        values = [
            perf_data['area']['total'],
            perf_data['timing']['critical_path_delay'],
            perf_data['power']['total_power_mw'],
            sum(perf_data['gate_counts'].values())
        ]
        
        bars = ax4.bar(metrics, values, color=['blue', 'green', 'red', 'orange'])
        ax4.set_title('Performance Summary')
        ax4.set_ylabel('Value')
        plt.setp(ax4.get_xticklabels(), rotation=45)
        
        # Add value labels
        for bar, value in zip(bars, values):
            ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
                    f'{value:.1f}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
    
    def plot_comparison_chart(self, design_stats_list, design_names):
        """Compare multiple designs"""
        if not design_stats_list:
            print("No design data for comparison")
            return
        
        metrics = ['Total Gates', 'Area', 'Delay (ps)', 'Power (mW)']
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        x = np.arange(len(design_names))
        width = 0.2
        
        # Extract metrics for each design
        for i, metric in enumerate(metrics):
            values = []
            for stats in design_stats_list:
                if metric == 'Total Gates':
                    values.append(sum(stats.get('gate_counts', {}).values()))
                elif metric == 'Area':
                    values.append(stats.get('area', {}).get('total', 0))
                elif metric == 'Delay (ps)':
                    values.append(stats.get('timing', {}).get('critical_path_delay', 0))
                elif metric == 'Power (mW)':
                    values.append(stats.get('power', {}).get('total_power_mw', 0))
            
            ax.bar(x + i*width, values, width, label=metric)
        
        ax.set_xlabel('Design')
        ax.set_ylabel('Value')
        ax.set_title('Design Comparison')
        ax.set_xticks(x + width*1.5)
        ax.set_xticklabels(design_names)
        ax.legend()
        
        plt.tight_layout()
        plt.show()
    
    def create_simple_circuit_diagram(self, gate_connections):
        """Create a simple circuit diagram using NetworkX"""
        G = nx.DiGraph()
        
        # Add nodes and edges from gate connections
        for connection in gate_connections:
            if 'input' in connection and 'output' in connection:
                G.add_edge(connection['input'], connection['output'])
        
        plt.figure(figsize=(12, 8))
        pos = nx.spring_layout(G)
        nx.draw(G, pos, with_labels=True, node_color='lightblue', 
                node_size=1500, font_size=10, font_weight='bold',
                arrows=True, arrowsize=20)
        plt.title("Simple Circuit Diagram")
        plt.show()

# Demonstrate visualizations
visualizer = DesignVisualizer()

print("DESIGN VISUALIZATION EXAMPLES")
print("="*35)

# Use results from previous performance analysis
if 'results' in locals():
    # Plot gate distribution
    visualizer.plot_gate_distribution(results['gate_counts'], "Sample Design Gate Distribution")
    
    # Plot comprehensive performance metrics
    visualizer.plot_performance_metrics(results)

# Create sample comparison data
sample_designs = [
    {'gate_counts': {'BUF_X1': 10, 'AND2_X1': 5}, 
     'area': {'total': 20}, 'timing': {'critical_path_delay': 150}, 
     'power': {'total_power_mw': 0.5}},
    {'gate_counts': {'BUF_X1': 15, 'OR2_X1': 8}, 
     'area': {'total': 25}, 'timing': {'critical_path_delay': 180}, 
     'power': {'total_power_mw': 0.7}},
]
design_names = ['Design A', 'Design B']

# visualizer.plot_comparison_chart(sample_designs, design_names)

# Simple circuit diagram example
sample_connections = [
    {'input': 'A', 'output': 'gate1'},
    {'input': 'B', 'output': 'gate1'},
    {'input': 'gate1', 'output': 'gate2'},
    {'input': 'C', 'output': 'gate2'},
    {'input': 'gate2', 'output': 'Y'}
]

# visualizer.create_simple_circuit_diagram(sample_connections)

print("Visualization tools ready! Run individual plot functions to see charts.")

## Hardware Trojan Analysis with ATTRITION

Now let's apply our tools to analyze the ATTRITION dataset and understand hardware Trojans in chip designs.

In [ ]:
# Hardware Trojan Detection and Analysis
class TrojanAnalyzer:
    """Analyze hardware Trojans using the ATTRITION dataset"""
    
    def __init__(self, analyzer, visualizer):
        self.analyzer = analyzer
        self.visualizer = visualizer
        
    def compare_original_vs_trojan(self, original_file, trojan_file, max_lines=5000):
        """Compare original design with Trojan-infected version"""
        print(f"COMPARING: {original_file} vs {trojan_file}")
        print("="*60)
        
        # Analyze original design
        original_stats = self.analyzer.parse_verilog_file(original_file, max_lines)
        trojan_stats = self.analyzer.parse_verilog_file(trojan_file, max_lines)
        
        if not original_stats or not trojan_stats:
            print("Error: Could not analyze one or both files")
            return None, None
        
        # Calculate differences
        differences = {
            'size_increase_kb': trojan_stats['file_size_kb'] - original_stats['file_size_kb'],
            'gate_increase': trojan_stats['gate_instances'] - original_stats['gate_instances'],
            'wire_increase': trojan_stats['wire_declarations'] - original_stats['wire_declarations'],
            'size_increase_percent': ((trojan_stats['file_size_kb'] - original_stats['file_size_kb']) / original_stats['file_size_kb']) * 100
        }
        
        print(f"Original Design Stats:")
        print(f"  File size: {original_stats['file_size_kb']:.1f} KB")
        print(f"  Gate instances: {original_stats['gate_instances']}")
        print(f"  Wire declarations: {original_stats['wire_declarations']}")
        
        print(f"\nTrojan-Infected Design Stats:")
        print(f"  File size: {trojan_stats['file_size_kb']:.1f} KB")
        print(f"  Gate instances: {trojan_stats['gate_instances']}")
        print(f"  Wire declarations: {trojan_stats['wire_declarations']}")
        
        print(f"\nTrojan Impact:")
        print(f"  Size increase: +{differences['size_increase_kb']:.1f} KB ({differences['size_increase_percent']:.2f}%)")
        print(f"  Gate increase: +{differences['gate_increase']}")
        print(f"  Wire increase: +{differences['wire_increase']}")
        
        return original_stats, trojan_stats, differences
    
    def analyze_trojan_distribution(self, design_name="aes_final", max_trojans=10):
        """Analyze multiple Trojan variants of a design"""
        trojan_dir = f"HT_infested_designs/{design_name}_set1"
        original_file = f"original_files/{design_name}.v"
        
        if not os.path.exists(trojan_dir) or not os.path.exists(original_file):
            print(f"Files not found for {design_name}")
            return
        
        print(f"TROJAN DISTRIBUTION ANALYSIS: {design_name}")
        print("="*50)
        
        # Get list of Trojan files
        trojan_files = [f for f in os.listdir(trojan_dir) if f.endswith('.v')][:max_trojans]
        
        # Analyze original
        original_stats = self.analyzer.parse_verilog_file(original_file, max_lines=3000)
        if not original_stats:
            print("Could not analyze original file")
            return
        
        trojan_analyses = []
        
        for trojan_file in trojan_files:
            trojan_path = os.path.join(trojan_dir, trojan_file)
            trojan_stats = self.analyzer.parse_verilog_file(trojan_path, max_lines=3000)
            
            if trojan_stats:
                impact = {
                    'filename': trojan_file,
                    'size_increase_percent': ((trojan_stats['file_size_kb'] - original_stats['file_size_kb']) / original_stats['file_size_kb']) * 100,
                    'gate_increase': trojan_stats['gate_instances'] - original_stats['gate_instances'],
                    'wire_increase': trojan_stats['wire_declarations'] - original_stats['wire_declarations']
                }
                trojan_analyses.append(impact)
        
        # Summary statistics
        if trojan_analyses:
            size_increases = [t['size_increase_percent'] for t in trojan_analyses]
            gate_increases = [t['gate_increase'] for t in trojan_analyses]
            
            print(f"Analyzed {len(trojan_analyses)} Trojan variants")
            print(f"Average size increase: {np.mean(size_increases):.2f}%")
            print(f"Size increase range: {min(size_increases):.2f}% to {max(size_increases):.2f}%")
            print(f"Average gate increase: {np.mean(gate_increases):.0f}")
            print(f"Gate increase range: {min(gate_increases)} to {max(gate_increases)}")
        
        return trojan_analyses
    
    def detect_suspicious_patterns(self, verilog_content):
        """Detect patterns that might indicate Trojans"""
        suspicious_patterns = []
        lines = verilog_content.split('\n')
        
        # Pattern indicators
        triggers = ['trigger', 'secret', 'hidden', 'unused', 'payload']
        rare_gates = ['XOR3', 'AOI', 'OAI', 'MUX4', 'LATCH']
        
        for i, line in enumerate(lines[:1000], 1):  # Check first 1000 lines
            line_lower = line.lower()
            
            # Check for suspicious signal names
            for trigger in triggers:
                if trigger in line_lower and ('wire' in line_lower or 'reg' in line_lower):
                    suspicious_patterns.append({
                        'type': 'suspicious_signal',
                        'line': i,
                        'pattern': trigger,
                        'text': line.strip()
                    })
            
            # Check for rare gate usage
            for gate in rare_gates:
                if gate in line:
                    suspicious_patterns.append({
                        'type': 'rare_gate',
                        'line': i,
                        'pattern': gate,
                        'text': line.strip()
                    })
        
        return suspicious_patterns

# Initialize Trojan analyzer
trojan_analyzer = TrojanAnalyzer(analyzer, visualizer)

# Example analysis (if files exist)
print("HARDWARE TROJAN ANALYSIS TOOLS READY")
print("="*45)

# Try to analyze AES design if available
if os.path.exists("original_files/aes_final.v"):
    print("Found AES design files - running analysis...")
    # trojan_analysis = trojan_analyzer.analyze_trojan_distribution("aes_final", max_trojans=5)
else:
    print("AES design files not found in current directory")
    print("This analysis would work when running in the ATTRITION project directory")

# Demonstrate suspicious pattern detection
sample_suspicious_code = """
wire secret_trigger;
wire hidden_payload;
XOR3_X1 rare_gate (.A(a), .B(b), .C(c), .Z(suspicious_out));
AOI21_X1 another_rare (.A1(d), .A2(e), .B(f), .ZN(payload_out));
"""

suspicious_findings = trojan_analyzer.detect_suspicious_patterns(sample_suspicious_code)
print(f"\nSuspicious Pattern Detection Example:")
print(f"Found {len(suspicious_findings)} suspicious patterns:")
for finding in suspicious_findings:
    print(f"  Line {finding['line']}: {finding['type']} - {finding['pattern']}")

## Interactive Exercises and Course Activities

Let's create some interactive exercises for students to practice hardware security concepts and LLM applications in chip design.

In [ ]:
# Interactive Exercises for Students
class CourseExercises:
    """Interactive exercises for LLM4ChipDesign course"""
    
    def __init__(self):
        self.exercises = {}
        self.current_exercise = None
    
    def exercise_1_verilog_generation(self):
        """Exercise 1: Generate and validate Verilog code"""
        print("="*60)
        print("EXERCISE 1: VERILOG CODE GENERATION AND VALIDATION")
        print("="*60)
        print()
        print("Task: Generate Verilog code for different modules and validate them")
        print()
        print("Instructions:")
        print("1. Use the VerilogGenerator to create different module types")
        print("2. Validate each generated module using VerilogValidator")
        print("3. Analyze the complexity metrics for each module")
        print("4. Compare the results")
        print()
        print("Try generating these modules:")
        print("- A 4-bit adder")
        print("- A 4-to-1 multiplexer")
        print("- A simple finite state machine")
        print()
        print("Example starter code:")
        print("```python")
        print("# Generate adder")
        print("adder_code = verilog_gen.get_template_code('adder')")
        print("validation = validator.check_syntax(adder_code)")
        print("complexity = validator.analyze_complexity(adder_code)")
        print("print(f'Valid: {validation[\"is_valid\"]}, Complexity: {complexity[\"complexity_score\"]}')")
        print("```")
        
        return {
            'task': 'verilog_generation',
            'tools': ['VerilogGenerator', 'VerilogValidator'],
            'expected_output': 'Validation results for 3 different modules'
        }
    
    def exercise_2_design_comparison(self):
        """Exercise 2: Compare hardware designs"""
        print("="*60)
        print("EXERCISE 2: HARDWARE DESIGN COMPARISON")
        print("="*60)
        print()
        print("Task: Compare different hardware designs using performance metrics")
        print()
        print("Instructions:")
        print("1. Create or load 2-3 different Verilog designs")
        print("2. Analyze each design using PerformanceMetrics")
        print("3. Create comparison visualizations")
        print("4. Identify which design is better for different criteria")
        print()
        print("Comparison criteria:")
        print("- Area efficiency")
        print("- Power consumption")
        print("- Timing performance")
        print("- Gate count")
        print()
        print("Example starter code:")
        print("```python")
        print("# Analyze multiple designs")
        print("designs = [adder_code, mux_code, fsm_code]")
        print("results = []")
        print("for i, design in enumerate(designs):")
        print("    analysis = perf_metrics.comprehensive_analysis(design)")
        print("    results.append(analysis)")
        print("visualizer.plot_comparison_chart(results, ['Adder', 'Mux', 'FSM'])")
        print("```")
        
        return {
            'task': 'design_comparison',
            'tools': ['PerformanceMetrics', 'DesignVisualizer'],
            'expected_output': 'Comparison charts and analysis report'
        }
    
    def exercise_3_trojan_detection(self):
        """Exercise 3: Hardware Trojan detection"""
        print("="*60)
        print("EXERCISE 3: HARDWARE TROJAN DETECTION")
        print("="*60)
        print()
        print("Task: Detect and analyze hardware Trojans in chip designs")
        print()
        print("Instructions:")
        print("1. Create a clean hardware design")
        print("2. Manually insert a simple 'Trojan' (extra gates/logic)")
        print("3. Use TrojanAnalyzer to detect differences")
        print("4. Analyze the impact on performance metrics")
        print()
        print("Trojan insertion ideas:")
        print("- Add extra AND/OR gates")
        print("- Insert unused wire declarations")
        print("- Add suspicious signal names")
        print("- Include rare gate types")
        print()
        print("Example starter code:")
        print("```python")
        print("# Create clean design")
        print("clean_design = verilog_gen.get_template_code('adder')")
        print()
        print("# Create 'infected' version")
        print("infected_design = clean_design + '''")
        print("wire secret_trigger;")
        print("AND2_X1 trojan_gate (.A1(secret_trigger), .A2(a), .Z(payload));")
        print("'''")
        print()
        print("# Analyze differences")
        print("suspicious = trojan_analyzer.detect_suspicious_patterns(infected_design)")
        print("```")
        
        return {
            'task': 'trojan_detection',
            'tools': ['TrojanAnalyzer', 'VerilogAnalyzer'],
            'expected_output': 'List of detected suspicious patterns'
        }
    
    def exercise_4_llm_assisted_design(self):
        """Exercise 4: LLM-assisted chip design"""
        print("="*60)
        print("EXERCISE 4: LLM-ASSISTED CHIP DESIGN")
        print("="*60)
        print()
        print("Task: Use LLM to assist in hardware design tasks")
        print()
        print("Instructions:")
        print("1. Create prompts for different design challenges")
        print("2. Generate design suggestions using the LLM")
        print("3. Validate and refine the generated designs")
        print("4. Compare LLM-generated vs template-based designs")
        print()
        print("Design challenges:")
        print("- Create a counter circuit")
        print("- Design a simple ALU component")
        print("- Generate a memory controller interface")
        print()
        print("Example starter code:")
        print("```python")
        print("# Create design prompt")
        print("prompt = 'Generate a Verilog module for a 4-bit binary counter'")
        print("generated_code = verilog_gen.generate_code(prompt)")
        print("validation = validator.check_syntax(generated_code)")
        print("print(f'Generated design validation: {validation}')")
        print("```")
        
        return {
            'task': 'llm_assisted_design',
            'tools': ['VerilogGenerator', 'LLM pipeline'],
            'expected_output': 'LLM-generated designs with validation results'
        }
    
    def show_all_exercises(self):
        """Display all available exercises"""
        exercises = [
            self.exercise_1_verilog_generation,
            self.exercise_2_design_comparison,
            self.exercise_3_trojan_detection,
            self.exercise_4_llm_assisted_design
        ]
        
        print("LLM4CHIPDESIGN COURSE EXERCISES")
        print("="*40)
        print()
        
        for i, exercise_func in enumerate(exercises, 1):
            print(f"EXERCISE {i}:")
            print("-" * 20)
            exercise_func()
            print()
        
        print("To start an exercise, run the corresponding function:")
        print("- exercises.exercise_1_verilog_generation()")
        print("- exercises.exercise_2_design_comparison()")
        print("- exercises.exercise_3_trojan_detection()")
        print("- exercises.exercise_4_llm_assisted_design()")

# Initialize course exercises
exercises = CourseExercises()

# Show all available exercises
exercises.show_all_exercises()

## Summary and Next Steps

### What You've Learned

In this notebook, you've explored:

1. **Environment Setup**: How to configure Google Colab for hardware design analysis
2. **LLM Integration**: Loading and configuring language models for chip design tasks
3. **Hardware Analysis**: Tools for parsing and analyzing Verilog designs
4. **Code Generation**: Using LLMs to generate and validate hardware description code
5. **Design Validation**: Implementing syntax and structure checking for HDL
6. **Design Rules**: Basic design rule checking for hardware constraints
7. **Performance Metrics**: Analyzing area, timing, and power characteristics
8. **Visualization**: Creating charts and diagrams for design analysis
9. **Hardware Security**: Understanding and detecting hardware Trojans
10. **Practical Exercises**: Hands-on activities to reinforce learning

### Key Tools Developed

- `VerilogAnalyzer`: Parse and analyze Verilog design files
- `VerilogGenerator`: Generate HDL code using LLM assistance
- `VerilogValidator`: Validate syntax and structure of generated code
- `DesignRuleChecker`: Check basic design constraints and rules
- `PerformanceMetrics`: Analyze area, timing, and power metrics
- `DesignVisualizer`: Create visualizations for design analysis
- `TrojanAnalyzer`: Detect and analyze hardware security threats

### Applications in Industry

These techniques are used in:
- **EDA Tools**: Electronic Design Automation software
- **Design Verification**: Automated testing and validation
- **Security Analysis**: Hardware Trojan detection and prevention
- **Design Optimization**: Performance and power optimization
- **Code Generation**: Automated HDL generation from specifications

### Further Learning

Continue your journey in LLM4ChipDesign by exploring:
- Advanced LLM fine-tuning for hardware-specific tasks
- Integration with commercial EDA tools
- Real-world hardware security case studies
- Machine learning for design optimization
- Formal verification methods

### Course Completion

Congratulations! You've completed the LLM4ChipDesign ATTRITION analysis notebook. You now have practical tools for applying LLM techniques to hardware design and security analysis.